In [2]:
import numpy as np, pandas as pd, os, zipfile
from datetime import datetime, timedelta, UTC

rng = np.random.default_rng(42)

# Demo settings (fast + convincing)
N_USERS, N_ITEMS, N_SCENES = 800, 70, 30
SLOTS_PER_SCENE = 2
N_EXPOSURES = 80_000
N_INTERACTIONS_TARGET = 50_000

OUT_DIR = "dynamic_vpp_synth_dataset"
os.makedirs(OUT_DIR, exist_ok=True)

countries = ["US","IN","EU","AE"]; country_probs = [0.35,0.35,0.20,0.10]
age_buckets = ["13-17","18-24","25-34","35-44","45+"]; age_probs = [0.12,0.33,0.27,0.16,0.12]
langs = {"US":["en"],"IN":["en","hi","te","ta"],"EU":["en","fr","de","es"],"AE":["en","ar"]}
genres = ["action","drama","comedy","romance","thriller"]; genre_probs=[0.25,0.25,0.20,0.15,0.15]
sens_levels=["low","medium","high"]; sens_probs=[0.55,0.35,0.10]
event_types=["view","click","add_to_cart","purchase"]; event_probs=[0.70,0.20,0.07,0.03]
event_w={"view":1.0,"click":3.0,"add_to_cart":4.0,"purchase":5.0}

D=5  # [sweet, premium, local, health, alcohol_affinity]
def make_id(p,i,w): return f"{p}{i:0{w}d}"
def pick(vals, probs, n): return [vals[i] for i in rng.choice(len(vals), size=n, p=probs)]
def sigmoid(x): return 1/(1+np.exp(-x))
def minor(ab): return ab=="13-17"

# Users
user_ids=[make_id("U",i,5) for i in range(1,N_USERS+1)]
u_country=pick(countries,country_probs,N_USERS)
u_age=pick(age_buckets,age_probs,N_USERS)
u_lang=[rng.choice(langs[c]).item() for c in u_country]
u_price=pick(["budget","mid","premium"],[0.45,0.40,0.15],N_USERS)
u_local=np.clip(rng.normal(0.55,0.22,N_USERS),0,1)
u_avoid=[bool(rng.random() < (0.85 if ab=="13-17" else 0.18)) for ab in u_age]

U_lat=rng.normal(0,1,(N_USERS,D))
U_lat[:,1]+=np.array([0.8 if p=="premium" else 0.0 for p in u_price])
U_lat[:,2]+=1.1*u_local
U_lat[:,4]+=np.array([-1.5 if a else 0.2 for a in u_avoid])

users=pd.DataFrame({
    "user_id":user_ids,"country":u_country,"age_bucket":u_age,"language":u_lang,
    "price_preference":u_price,"local_brand_affinity":np.round(u_local,3),"avoid_alcohol":u_avoid
})

# Items
item_ids=[make_id("B",i,4) for i in range(1,N_ITEMS+1)]
cats=["cola","soda","water","juice","energy_drink","beer"]; cat_probs=[0.20,0.18,0.20,0.16,0.14,0.12]
i_cat=pick(cats,cat_probs,N_ITEMS)
i_alc=[c=="beer" for c in i_cat]
i_minage=[21 if a else 0 for a in i_alc]
i_price=pick(["budget","mid","premium"],[0.50,0.35,0.15],N_ITEMS)
i_pack=pick(["glass_bottle","can","plastic_bottle"],[0.55,0.35,0.10],N_ITEMS)

i_avail=[]; i_origin=[]
for c in i_cat:
    if c=="beer":
        if rng.random()<0.55:
            r=rng.choice(["US","EU","AE"]).item(); i_avail.append(r); i_origin.append("local")
        else:
            i_avail.append("US,EU"); i_origin.append("global")
    else:
        if rng.random()<0.45:
            r=rng.choice(countries).item(); i_avail.append(r); i_origin.append("local")
        else:
            i_avail.append("US,IN,EU,AE"); i_origin.append("global")

I_attr=rng.normal(0,1,(N_ITEMS,D))
for i,c in enumerate(i_cat):
    if c in ("cola","soda","energy_drink"): I_attr[i,0]+=1.0
    if c in ("water","juice"): I_attr[i,3]+=1.0
    if c=="beer": I_attr[i,4]+=1.5
I_attr[:,1]+=np.array([0.9 if p=="premium" else (0.2 if p=="mid" else -0.3) for p in i_price])
I_attr[:,2]+=np.array([1.1 if o=="local" else -0.2 for o in i_origin])

name_base={"cola":"Cola","soda":"Soda","water":"Spring","juice":"Juice","energy_drink":"Energy","beer":"Brew"}
i_names=[f"{name_base[c]}{k}" for k,c in enumerate(i_cat, start=1)]

items=pd.DataFrame({
    "item_id":item_ids,"brand_name":i_names,"category":i_cat,"is_alcohol":i_alc,"min_age":i_minage,
    "price_tier":i_price,"region_availability":i_avail,"origin":i_origin,"packaging":i_pack
})

# Scenes + slots
scene_ids=[make_id("S",i,3) for i in range(1,N_SCENES+1)]
scenes=pd.DataFrame({
    "scene_id":scene_ids,
    "genre":pick(genres,genre_probs,N_SCENES),
    "sensitivity":pick(sens_levels,sens_probs,N_SCENES),
    "mood":pick(["tense","calm","happy","serious"],[0.25,0.25,0.25,0.25],N_SCENES)
})

slots=[]; sid=1
for sc in scene_ids:
    for _ in range(SLOTS_PER_SCENE):
        slots.append({
            "scene_id":sc,"slot_id":f"SL{sid:04d}","placement_object":"bottle",
            "visibility":rng.choice(["background","midground","foreground"], p=[0.45,0.35,0.20]).item(),
            "occlusion":rng.choice(["low","medium","high"], p=[0.55,0.35,0.10]).item(),
            "motion":rng.choice(["static","moving"], p=[0.65,0.35]).item()
        })
        sid+=1
placement_slots=pd.DataFrame(slots)

# Exposures + interactions
start_time=datetime(2026,1,1,12,0,0, tzinfo=UTC)
def rand_time():
    return (start_time + timedelta(days=int(rng.integers(0,30)), minutes=int(rng.integers(0,1440)))).isoformat()

def allowed(it,u,sens):
    if it["is_alcohol"] and (minor(u["age_bucket"]) or u["avoid_alcohol"] or sens=="high"):
        return False
    avail=it["region_availability"].split(",")
    if u["country"] not in avail and it["region_availability"]!="US,IN,EU,AE":
        return False
    return True

pop_bias=rng.normal(0,0.35,N_ITEMS)

u_idx=rng.integers(0,N_USERS,size=N_EXPOSURES)
s_idx=rng.integers(0,N_SCENES,size=N_EXPOSURES)
sl_idx=rng.integers(0,len(placement_slots),size=N_EXPOSURES)
i_idx=rng.integers(0,N_ITEMS,size=N_EXPOSURES)

expo=[]; inter=[]
for n in range(N_EXPOSURES):
    ui,si,sli,ii=int(u_idx[n]),int(s_idx[n]),int(sl_idx[n]),int(i_idx[n])
    u=users.iloc[ui]; it=items.iloc[ii]; sc=scenes.iloc[si]; sl=placement_slots.iloc[sli]

    score=float(np.dot(U_lat[ui], I_attr[ii]) + pop_bias[ii])
    if it["origin"]=="local" and u["local_brand_affinity"]>0.6 and u["country"] in it["region_availability"].split(","):
        score+=0.6
    if it["is_alcohol"] and sc["sensitivity"] in ("medium","high"):
        score-=0.9

    engaged = rng.random() < sigmoid(score/2.2)
    ok = allowed(it,u,sc["sensitivity"])

    outcome="no_action"
    if not ok:
        hide_p=0.55 if (minor(u["age_bucket"]) or u["avoid_alcohol"]) else 0.25
        if it["is_alcohol"] and rng.random()<hide_p:
            outcome="hide_ad"
    elif engaged:
        et=rng.choice(event_types,p=event_probs).item()
        outcome=et
        inter.append({
            "user_id":u["user_id"],"item_id":it["item_id"],"event_type":et,"rating":event_w[et],
            "timestamp":rand_time(),"scene_id":sc["scene_id"],"slot_id":sl["slot_id"],
            "country":u["country"],"age_bucket":u["age_bucket"],
            "content_sensitivity":sc["sensitivity"],"placement_object":"bottle"
        })

    expo.append({
        "user_id":u["user_id"],"item_id":it["item_id"],"scene_id":sc["scene_id"],"slot_id":sl["slot_id"],
        "shown":1,"allowed_by_rules":int(ok),"outcome":outcome,"timestamp":rand_time(),
        "country":u["country"],"age_bucket":u["age_bucket"],"content_sensitivity":sc["sensitivity"]
    })

exposures=pd.DataFrame(expo)
interactions=pd.DataFrame(inter)
if len(interactions)>N_INTERACTIONS_TARGET:
    interactions=interactions.sample(n=N_INTERACTIONS_TARGET, random_state=42).reset_index(drop=True)

# Save CSVs
files = {
    "users.csv": users,
    "items.csv": items,
    "scenes.csv": scenes,
    "placement_slots.csv": placement_slots,
    "exposures.csv": exposures,
    "interactions.csv": interactions,
}
for fn, df in files.items():
    df.to_csv(os.path.join(OUT_DIR, fn), index=False)

# README
readme_path = os.path.join(OUT_DIR, "README.txt")
with open(readme_path, "w") as f:
    f.write(
        "Synthetic dataset for CAPS demo (dynamic virtual product placement)\n"
        f"Generated: {datetime.now(UTC).isoformat()}\n\n"
        f"users: {len(users)}\nitems: {len(items)}\nscenes: {len(scenes)}\nslots: {len(placement_slots)}\n"
        f"exposures: {len(exposures)}\ninteractions: {len(interactions)}\n"
    )

# Zip
zip_name = "dynamic_vpp_synth_dataset.zip"
with zipfile.ZipFile(zip_name, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for fn in list(files.keys()) + ["README.txt"]:
        z.write(os.path.join(OUT_DIR, fn), arcname=fn)

print("Created:", zip_name)


Created: dynamic_vpp_synth_dataset.zip
